# High-z Example 05: V/sigma Kinematic State Profiles

**EPS Research — High-z Kinematic Corpus Z1**

V/sigma = Vrot/sigma per ring measures the kinematic state:
- V/sigma > 3: rotation-dominated (disk-like)
- V/sigma < 1: dispersion-dominated

At z~5, even confirmed rotators show lower V/sigma than local disks,
reflecting the elevated turbulence in high-z star-forming galaxies.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, g in enumerate(sorted(rotators, key=lambda x: x['redshift'])):
    d   = g['data']
    R   = [p['R_kpc']       for p in d]
    vos = [p['v_over_sigma'] for p in d]
    ax  = axes[i]
    ax.plot(R, vos, 'o-', color='#9b59b6', linewidth=1.5, markersize=5)
    ax.axhline(3.0, color='green',  ls='--', lw=1, alpha=0.7, label='V/σ=3 (W15)')
    ax.axhline(1.0, color='orange', ls='--', lw=1, alpha=0.7, label='V/σ=1')
    ax.set_title(f"{g['galaxy']}  z={g['redshift']:.4f}", fontsize=8)
    ax.set_xlabel('R (kpc)', fontsize=7)
    ax.set_ylabel('V/σ', fontsize=7)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=6)

plt.suptitle('V/σ Kinematic State — All 8 Z1 Tier-1 Rotators\n'
             'EPS Research Z1 Corpus | Jones et al. (2021)', fontsize=11)
plt.tight_layout()
plt.savefig('hz05_v_sigma_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
vos_outer = [g['data'][-1]['v_over_sigma'] for g in rotators]
print(f"V/sigma at outermost ring:")
print(f"  Range:  {min(vos_outer):.2f} -- {max(vos_outer):.2f}")
print(f"  Median: {np.median(vos_outer):.2f}")
print(f"  Above W15 threshold (V/σ>3): {sum(1 for v in vos_outer if v>3)}/8")